### description

load,clean,split

In [36]:
        import pandas as pd
        import numpy as np
        import re

        from sklearn.model_selection import train_test_split
        from sklearn.metrics import classification_report

        df=pd.read_csv('datasets/dataset1_text_rich_transactions_harder.csv')
        df['transaction_date']=pd.to_datetime(df['transaction_date'],format="%d-%m-%Y",errors="coerce")
        df = df.dropna(subset=["description", "category_label", "transaction_date"])


        # Text cleaning: robust regex-based normalization
        def clean_text(s: str) -> str:
            if not isinstance(s, str):
                return ""
            s = s.lower()
            # remove common bank noise tokens you don't want to dominate
            noise_tokens = [
                r"\bfy\d{2}\b",          # fy24, fy25
                r"\bq[1-4]\b",           # q1, q2
                r"\binv/?\d*\b",         # inv, inv/2404/001
                r"\bbill/?\d*\b",        # bill/...
                r"\brcpt/?\d*\b",        # rcpt/...
                r"\btaxinv/?\d*\b",      # taxinv/...
                r"\bref\b\s*\d+",        # ref 1234
                r"\badv\b",              # adv
                r"\bsubs\b",             # subs
                r"\bgst\b",              # gst word (optional)
            ]
            for pat in noise_tokens:
                s = re.sub(pat, " ", s)
            # keep alphanumerics and basic separators
            s = re.sub(r"[^a-z0-9\s]", " ", s)
            # collapse spaces
            s = re.sub(r"\s+", " ", s).strip()
            return s

        df["description_clean"] = df["description"].apply(clean_text)


        # normalize vendor_name for some models
        # def clean_vendor(v):
        #     if not isinstance(v, str):
        #         return ""
        #     v = v.upper()
        #     v = re.sub(r"\bPVT\.?\b|\bLTD\.?\b|\bLIMITED\b|\bINDIA\b", " ", v)
        #     v = re.sub(r"[^A-Z0-9\s]", " ", v)
        #     v = re.sub(r"\s+", " ", v).strip()
        #     return v

        # df["vendor_clean"] = df["vendor_name"].apply(clean_vendor)


        df["month"] = df["transaction_date"].dt.month
        df["dow"] = df["transaction_date"].dt.dayofweek  # 0=Monday


        TEXT_COL = "description_clean"
        # VENDOR_COL = "vendor_clean"
        NUM_COLS = ["amount", "month", "dow"]
        TARGET_COL = "category_label"
    
        X_text = df[[TEXT_COL] + NUM_COLS]
        y = df[TARGET_COL]


        X_train, X_test, y_train, y_test = train_test_split(
            X_text, y, test_size=0.2, stratify=y, random_state=42
        )

In [37]:
df.head()

,transaction_id,transaction_date,amount,currency,description,vendor_name,gst_applicable,gst_slab,itc_eligible,category_label,is_anomaly,description_clean,month,dow
0,TXN0000001,2024-04-01,7641.92,INR,imhs ikea rcpt/2404/001 subs - exp,IKEA,True,12%,TRUE,Office Supplies,0,imhs ikea 001 exp,4,0
1,TXN0000002,2024-04-01,58560.14,INR,upi awfis rcpt/2404/002 fy24 - rent exp# ref ...,Awfis,True,18%,TRUE,Rent,0,upi awfis 002 rent exp,4,0
2,TXN0000003,2024-04-01,2184.95,INR,ZOMATO,ZOMATO,True,5%,FALSE,Meals,0,zomato,4,0
3,TXN0000004,2024-04-01,5375.79,INR,*CARD OFFICE DEPOT Pvt Ltd INV/2404/004 FY25 -...,OFFICE DEPOT Pvt Ltd,True,12%,TRUE,Office Supplies,0,card office depot pvt ltd 004 exp,4,0
5,TXN0000006,2024-04-01,11191.62,INR,/ card kseb rcpt/2404/006 rent - exp,KSEB,True,18%,TRUE,Utilities,0,card kseb 006 rent exp,4,0


TF‑IDF + Logistic Regression


In [38]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression


# # Build combined text feature: description + vendor
# for df_part in (X_train, X_test):
#     df_part["text_combo"] = (
#         df_part[TEXT_COL].fillna("") + " " + df_part[VENDOR_COL].fillna("")
#     )
    
    
# TEXT_COMBO = "text_combo"

tfidf = TfidfVectorizer(
    max_features=8000,          # high enough for variety, small enough for speed
    ngram_range=(1, 2),         # unigrams + bigrams
    min_df=3,                   # drop very rare terms
)



#Preprocessor: TF-IDF on text, scaling on numeric
preprocess_lr = ColumnTransformer(
    transformers=[
        ("text", tfidf, TEXT_COL),
        ("num", StandardScaler(), NUM_COLS),
    ],
    remainder="drop",
)


# Logistic Regression (multinomial)
log_reg = LogisticRegression(
    max_iter=2000,
    n_jobs=-1,
    multi_class="multinomial",
    class_weight="balanced"     # helpful for class imbalance
)



pipe_lr = Pipeline(
    steps=[
        ("preprocess", preprocess_lr),
        ("clf", log_reg),
    ]
)


pipe_lr.fit(X_train, y_train)





c:\ProgramData\anaconda3\envs\exassaro\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('text',
                                                  TfidfVectorizer(max_features=8000,
                                                                  min_df=3,
                                                                  ngram_range=(1,
                                                                               2)),
                                                  'description_clean'),
                                                 ('num', StandardScaler(),
                                                  ['amount', 'month',
                                                   'dow'])])),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=2000,
                                    multi_class='multinomial', n_jobs=-1))])

In [39]:
y_pred_lr = pipe_lr.predict(X_test)
print(" TF-IDF + Logistic Regression ")
print(classification_report(y_test, y_pred_lr))

 TF-IDF + Logistic Regression 
                 precision    recall  f1-score   support

Exempt Services       0.65      0.92      0.76        26
    IT Services       0.99      0.95      0.97       216
          Meals       0.92      0.97      0.95        74
Office Supplies       0.99      0.95      0.97       124
           Rent       0.97      0.99      0.98       151
       Software       0.98      0.97      0.98       115
       Training       0.98      0.96      0.97        49
         Travel       1.00      0.94      0.97        82
      Utilities       0.94      0.96      0.95       143

       accuracy                           0.96       980
      macro avg       0.94      0.96      0.94       980
   weighted avg       0.96      0.96      0.96       980



TF‑IDF + XGBoost (or LightGBM)


In [40]:
from sklearn.preprocessing import LabelEncoder

# y is currently category_label as strings
le = LabelEncoder()
y_encoded = le.fit_transform(y)          # maps classes to 0..n-1

X_train, X_test, y_train, y_test = train_test_split(
    X_text, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42
)


In [41]:
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline

tfidf_xgb = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=3,
)

xgb_clf = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    tree_method="hist",
    max_depth=8,
    learning_rate=0.1,
    n_estimators=300,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
)

pipe_xgb = Pipeline(
    steps=[
        ("tfidf", tfidf_xgb),
        ("clf", xgb_clf),
    ]
)

pipe_xgb.fit(X_train["description_clean"], y_train)
y_pred_xgb = pipe_xgb.predict(X_test["description_clean"])

print("=== TF-IDF + XGBoost (description only) ===")
print(classification_report(le.inverse_transform(y_test),
                            le.inverse_transform(y_pred_xgb)))


=== TF-IDF + XGBoost (description only) ===
                 precision    recall  f1-score   support

Exempt Services       0.48      0.54      0.51        26
    IT Services       0.91      0.96      0.94       216
          Meals       0.99      0.93      0.96        74
Office Supplies       0.97      0.94      0.96       124
           Rent       0.98      0.94      0.96       151
       Software       0.98      0.97      0.97       115
       Training       1.00      0.96      0.98        49
         Travel       0.99      0.95      0.97        82
      Utilities       0.92      0.96      0.94       143

       accuracy                           0.94       980
      macro avg       0.91      0.91      0.91       980
   weighted avg       0.94      0.94      0.94       980



In [42]:
from lightgbm import LGBMClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

# LightGBM uses the same TF-IDF representation on description_clean
tfidf_lgbm = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=3,
)

lgbm_clf = LGBMClassifier(
    objective="multiclass",
    num_leaves=63,
    learning_rate=0.05,
    n_estimators=400,
    subsample=0.8,
    colsample_bytree=0.8,
)

pipe_lgbm = Pipeline(
    steps=[
        ("tfidf", tfidf_lgbm),
        ("clf", lgbm_clf),
    ]
)

# Train on description only (same as XGBoost)
pipe_lgbm.fit(X_train["description_clean"], y_train)
y_pred_lgbm = pipe_lgbm.predict(X_test["description_clean"])

print("=== TF-IDF + LightGBM (description only) ===")
print(classification_report(
    le.inverse_transform(y_test),
    le.inverse_transform(y_pred_lgbm)
))


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000972 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6020
[LightGBM] [Info] Number of data points in the train set: 3919, number of used features: 180
[LightGBM] [Info] Start training from score -3.610153
[LightGBM] [Info] Start training from score -1.512019
[LightGBM] [Info] Start training from score -2.576498
[LightGBM] [Info] Start training from score -2.065002
[LightGBM] [Info] Start training from score -1.868363
[LightGBM] [Info] Start training from score -2.148908
[LightGBM] [Info] Start training from score -3.005734
[LightGBM] [Info] Start training from score -2.486694
[LightGBM] [Info] Start training from score -1.924453
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] 

c:\ProgramData\anaconda3\envs\exassaro\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


=== TF-IDF + LightGBM (description only) ===
                 precision    recall  f1-score   support

Exempt Services       0.44      0.42      0.43        26
    IT Services       0.89      0.96      0.92       216
          Meals       0.93      0.88      0.90        74
Office Supplies       0.89      0.95      0.92       124
           Rent       0.96      0.95      0.95       151
       Software       0.94      0.90      0.92       115
       Training       0.95      0.84      0.89        49
         Travel       0.96      0.90      0.93        82
      Utilities       0.96      0.94      0.95       143

       accuracy                           0.92       980
      macro avg       0.88      0.86      0.87       980
   weighted avg       0.92      0.92      0.92       980



In [43]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

bow_vec = CountVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=3,
)

nb_clf = MultinomialNB()

pipe_nb = Pipeline(
    steps=[
        ("bow", bow_vec),
        ("clf", nb_clf),
    ]
)

pipe_nb.fit(X_train["description_clean"], y_train)
y_pred_nb = pipe_nb.predict(X_test["description_clean"])

print("=== Multinomial Naive Bayes (description only) ===")
print(classification_report(le.inverse_transform(y_test),
                            le.inverse_transform(y_pred_nb)))


=== Multinomial Naive Bayes (description only) ===
                 precision    recall  f1-score   support

Exempt Services       1.00      0.35      0.51        26
    IT Services       0.87      0.97      0.91       216
          Meals       1.00      0.97      0.99        74
Office Supplies       1.00      0.95      0.98       124
           Rent       0.95      0.94      0.94       151
       Software       1.00      0.97      0.98       115
       Training       1.00      0.96      0.98        49
         Travel       1.00      0.95      0.97        82
      Utilities       0.90      0.97      0.93       143

       accuracy                           0.94       980
      macro avg       0.97      0.89      0.91       980
   weighted avg       0.95      0.94      0.94       980



In [44]:


from sentence_transformers import SentenceTransformer

# 1) Prepare plain text lists
train_texts = X_train["description_clean"].fillna("").tolist()
test_texts = X_test["description_clean"].fillna("").tolist()

# 2) Load a compact embedding model
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# 3) Compute embeddings for description only
X_train_emb = embed_model.encode(train_texts, batch_size=64, show_progress_bar=True)
X_test_emb = embed_model.encode(test_texts, batch_size=64, show_progress_bar=True)

# 4) XGBoost on embeddings
xgb_emb_clf = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    tree_method="hist",
    max_depth=8,
    learning_rate=0.1,
    n_estimators=300,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
)

xgb_emb_clf.fit(X_train_emb, y_train)
y_pred_emb = xgb_emb_clf.predict(X_test_emb)

print("=== Sentence embeddings + XGBoost (description only) ===")
print(classification_report(le.inverse_transform(y_test),
                            le.inverse_transform(y_pred_emb)))


Batches: 100%|██████████| 16/16 [00:02<00:00,  5.83it/s]


=== Sentence embeddings + XGBoost (description only) ===
                 precision    recall  f1-score   support

Exempt Services       1.00      0.54      0.70        26
    IT Services       0.91      0.94      0.93       216
          Meals       0.97      0.97      0.97        74
Office Supplies       1.00      0.97      0.98       124
           Rent       0.97      0.94      0.96       151
       Software       0.87      0.96      0.91       115
       Training       0.98      0.92      0.95        49
         Travel       0.97      0.94      0.96        82
      Utilities       0.91      0.97      0.94       143

       accuracy                           0.94       980
      macro avg       0.95      0.91      0.92       980
   weighted avg       0.94      0.94      0.94       980

